# 哈夫曼编码与算术编码实验：学生练习版

本实验实现哈夫曼编码与算术编码。练习版保留文本准备、频率统计、压缩率分析、结果打印和对比框架，将关键考查内容设置为 `TODO`，需要学生补全。

需要完成的核心内容：

1. 哈夫曼树构造。
2. 哈夫曼编码表生成。
3. 哈夫曼编码与解码。
4. 哈夫曼树绘图布局。
5. 算术编码符号区间构造。
6. 算术编码与解码。
7. 算术编码区间到二进制码字的转换。

## 1. 相关背景知识

### 1.1 无损压缩与变长编码

无损压缩要求编码后能够完全还原原始数据。对于文本、灰度图像像素或符号序列，如果某些符号出现频率较高，就可以给高频符号分配较短编码，给低频符号分配较长编码，从而减少总编码长度。

### 1.2 哈夫曼编码

哈夫曼编码是一种经典的前缀编码方法。它根据符号出现频率构造一棵二叉树：

1. 每个符号先作为一个叶子节点。
2. 每次取出频率最小的两个节点合并成一个新节点。
3. 新节点频率等于两个子节点频率之和。
4. 重复合并，直到只剩一个根节点。
5. 从根到叶子的路径就是该符号的编码，左分支记为 `0`，右分支记为 `1`。

哈夫曼编码具有 **前缀码** 特性：任何一个符号的编码都不是另一个符号编码的前缀，因此可以唯一解码。

### 1.3 算术编码

算术编码不是为每个符号单独分配一个固定码字，而是把整个消息编码为 `[0, 1)` 区间中的一个数。

基本思想：

1. 根据符号概率把 `[0, 1)` 划分为若干子区间。
2. 读入第一个符号后，把当前区间缩小到该符号对应的子区间。
3. 继续读入下一个符号，在当前区间内部再次按概率划分。
4. 所有符号处理完后，得到一个很小的区间。
5. 只要选择该区间内任意一个数，就可以表示整条消息。

算术编码通常能更接近信息熵下界，但实现时需要处理精度、区间缩放和二进制输出等问题。本实验使用 Python 标准库中的 `Fraction` 做精确有理数计算，便于学生理解区间变化过程。

## 2. 实验步骤

本实验按以下步骤完成：

1. 准备一段待编码文本。
2. 统计字符频率。
3. 实现哈夫曼编码：
   - 构建哈夫曼树。
   - 生成编码表。
   - 对文本进行编码。
   - 根据哈夫曼树进行解码。
4. 实现算术编码：
   - 根据频率构造累积概率区间。
   - 逐字符缩小编码区间。
   - 在最终区间中选择一个二进制码值。
   - 根据码值和消息长度完成解码。
5. 对比两种编码方法的编码长度和解码正确性。

## 学生练习任务与代码补全步骤

请按照下面顺序补全代码。建议每补完一个函数，就运行当前单元检查输出。

### 1. 补全哈夫曼树构造 `build_huffman_tree`

实现步骤：

1. 为每个字符创建一个 `HuffmanNode` 叶子节点。
2. 将节点按频率放入最小堆。
3. 如果只有一种字符，需要单独构造一个根节点。
4. 当堆中节点数量大于 1 时，重复执行：
   - 取出频率最小的两个节点。
   - 合并成一个内部节点。
   - 新节点频率等于两个子节点频率之和。
   - 将新节点重新放回最小堆。
5. 最后堆中剩余的节点就是哈夫曼树根节点。

### 2. 补全哈夫曼编码表 `generate_huffman_codes`

实现步骤：

1. 从根节点开始递归遍历。
2. 向左走时，在当前编码后追加 `"0"`。
3. 向右走时，在当前编码后追加 `"1"`。
4. 遇到叶子节点时，把当前编码存入 `codes`。
5. 如果整棵树只有一个叶子节点，可使用 `"0"` 作为编码。

### 3. 补全哈夫曼编码与解码

`huffman_encode`：

1. 遍历原始文本中的每个字符。
2. 从编码表 `codes` 中取出该字符的编码。
3. 将所有编码拼接成完整二进制字符串。

`huffman_decode`：

1. 从哈夫曼树根节点开始。
2. 读到 `0` 向左走，读到 `1` 向右走。
3. 到达叶子节点时输出该字符。
4. 回到根节点继续读取下一段编码。

### 4. 补全哈夫曼树绘图布局 `assign_huffman_tree_positions`

实现步骤：

1. 对树做递归遍历。
2. 叶子节点按照从左到右的顺序分配递增的 `x` 坐标。
3. 节点深度越深，`y` 坐标越小。
4. 内部节点的 `x` 坐标等于左右子节点 `x` 坐标的平均值。
5. 返回节点到坐标的映射字典 `positions`。

### 5. 补全算术编码区间 `build_arithmetic_intervals`

实现步骤：

1. 统计所有符号总频数。
2. 按固定顺序遍历符号，建议使用 `sorted(frequencies)`。
3. 将频率转换为概率 `frequency / total`。
4. 使用累积概率为每个符号分配 `[low, high)` 区间。
5. 区间端点使用 `Fraction`，避免浮点误差。

### 6. 补全算术编码 `arithmetic_encode`

实现步骤：

1. 初始区间为 `[0, 1)`。
2. 每读入一个符号，取出该符号在全局概率表中的 `[symbol_low, symbol_high)`。
3. 计算当前区间宽度 `current_width = high - low`。
4. 将当前区间缩小到该符号对应的子区间。
5. 全部符号处理完后，选择最终区间中点作为 `tag`。

### 7. 补全算术解码 `arithmetic_decode`

实现步骤：

1. 将 `tag` 看作当前待解码值。
2. 查找它落入哪个符号区间。
3. 输出该符号。
4. 将当前值归一化到该符号子区间内部：

$$
value = \frac{value-symbol\_low}{symbol\_high-symbol\_low}
$$

5. 重复 `message_length` 次。

### 8. 补全二进制码字选择 `find_binary_fraction_in_interval`

实现步骤：

1. 从较短的二进制长度开始尝试。
2. 对长度为 `bit_length` 的二进制小数，分母为 `2 ** bit_length`。
3. 找到第一个落入 `[low, high)` 的分数。
4. 将分子格式化为固定长度二进制字符串。
5. 返回二进制字符串和对应的 `Fraction` 码值。

## 3. 导入标准库

In [ ]:
from collections import Counter
from fractions import Fraction
import heapq
import itertools
import math

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 4. 准备待编码文本并统计频率

In [ ]:
sample_text = "AAAAAABBBBBCCCCDDDEEFFGGH_HUFFMAN_AND_ARITHMETIC_CODING"


def calculate_frequencies(text):
    """
    统计文本中每个字符出现的次数。

    参数：
        text: 输入字符串

    返回：
        frequencies: 字符到出现次数的字典
    """
    if not text:
        raise ValueError("输入文本不能为空")
    return dict(Counter(text))


def print_frequency_table(frequencies):
    """
    打印字符频率表。
    """
    total = sum(frequencies.values())
    print("字符 | 次数 | 概率")
    print("-" * 22)
    for symbol, count in sorted(frequencies.items(), key=lambda item: (-item[1], item[0])):
        shown = repr(symbol)
        probability = count / total
        print(f"{shown:>4} | {count:>4} | {probability:.4f}")


frequencies = calculate_frequencies(sample_text)

print("原始文本：")
print(sample_text)
print("\n文本长度：", len(sample_text))
print("\n频率表：")
print_frequency_table(frequencies)

## 5. 实现哈夫曼编码

In [ ]:
class HuffmanNode:
    """
    哈夫曼树节点。

    参数：
        symbol: 叶子节点对应的符号；内部节点为 None
        frequency: 节点频率
        left: 左子节点
        right: 右子节点
    """

    def __init__(self, symbol=None, frequency=0, left=None, right=None):
        self.symbol = symbol
        self.frequency = frequency
        self.left = left
        self.right = right

    def is_leaf(self):
        return self.left is None and self.right is None


def build_huffman_tree(frequencies):
    """
    TODO：根据频率表构建哈夫曼树。
    """
    # TODO 1：创建最小堆 heap 和计数器 counter。
    # TODO 2：遍历 frequencies，为每个符号创建 HuffmanNode，并压入 heap。
    # TODO 3：处理只有一个符号的特殊情况。
    # TODO 4：循环取出两个频率最小的节点，合并成内部节点。
    # TODO 5：将合并后的节点重新压入 heap。
    # TODO 6：返回最终根节点。
    raise NotImplementedError("请补全 build_huffman_tree 函数")


def generate_huffman_codes(root):
    """
    TODO：根据哈夫曼树生成编码表。
    """
    # TODO 1：创建空字典 codes。
    # TODO 2：定义递归函数 traverse(node, prefix)。
    # TODO 3：遇到叶子节点时，将 prefix 保存为该符号编码。
    # TODO 4：递归遍历左子树时追加 '0'。
    # TODO 5：递归遍历右子树时追加 '1'。
    # TODO 6：返回 codes。
    raise NotImplementedError("请补全 generate_huffman_codes 函数")


def huffman_encode(text, codes):
    """
    TODO：使用哈夫曼编码表对文本进行编码。
    """
    # TODO：遍历 text，将每个字符替换为 codes 中对应的二进制编码并拼接。
    raise NotImplementedError("请补全 huffman_encode 函数")


def huffman_decode(encoded_bits, root):
    """
    TODO：根据哈夫曼树对二进制字符串进行解码。
    """
    # TODO 1：从 root 开始扫描 encoded_bits。
    # TODO 2：遇到 '0' 走左子树，遇到 '1' 走右子树。
    # TODO 3：到达叶子节点时输出符号，并回到 root。
    # TODO 4：返回解码字符串。
    raise NotImplementedError("请补全 huffman_decode 函数")


huffman_root = build_huffman_tree(frequencies)
huffman_codes = generate_huffman_codes(huffman_root)
huffman_bits = huffman_encode(sample_text, huffman_codes)
huffman_decoded = huffman_decode(huffman_bits, huffman_root)

print("哈夫曼编码表：")
for symbol, bits in sorted(huffman_codes.items(), key=lambda item: (len(item[1]), item[0])):
    print(f"{repr(symbol):>4} -> {bits}")

print("\n哈夫曼编码结果：")
print(huffman_bits)
print("\n哈夫曼编码长度：", len(huffman_bits), "bit")
print("哈夫曼解码是否正确：", huffman_decoded == sample_text)

## 6. 绘制哈夫曼树

下面将前面构造出的 `huffman_root` 画成树形结构。图中：

1. 圆形节点表示哈夫曼树节点。
2. 叶子节点显示字符和频率，例如 `'A':9`。
3. 内部节点显示合并后的频率。
4. 左分支标记为 `0`，右分支标记为 `1`。

通过这张图可以直观看到：频率越高的字符通常离根节点越近，编码越短。

In [ ]:
def assign_huffman_tree_positions(root):
    """
    TODO：为哈夫曼树中的每个节点分配绘图坐标。
    """
    # TODO 1：创建 positions 字典。
    # TODO 2：创建 leaf_counter，用于给叶子节点分配从左到右的 x 坐标。
    # TODO 3：递归遍历哈夫曼树。
    # TODO 4：叶子节点直接分配坐标 (x, -depth)。
    # TODO 5：内部节点先递归处理左右子树，再将 x 设为左右子节点 x 坐标平均值。
    # TODO 6：返回 positions。
    raise NotImplementedError("请补全 assign_huffman_tree_positions 函数")


def huffman_node_label(node):
    """
    生成哈夫曼树节点显示文本。
    """
    if node.is_leaf():
        return f"{repr(node.symbol)}\n{node.frequency}"
    return str(node.frequency)


def draw_huffman_tree(root, figsize=(15, 7)):
    """
    使用 matplotlib 绘制哈夫曼树。
    """
    positions = assign_huffman_tree_positions(root)

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_title("哈夫曼树")
    ax.axis("off")

    def draw_edges(node):
        if node is None:
            return

        x0, y0 = positions[node]
        for child, bit in [(node.left, "0"), (node.right, "1")]:
            if child is None:
                continue
            x1, y1 = positions[child]
            ax.plot([x0, x1], [y0, y1], color="#4b5563", linewidth=1.4)
            ax.text(
                (x0 + x1) / 2,
                (y0 + y1) / 2 + 0.08,
                bit,
                ha="center",
                va="center",
                fontsize=10,
                color="#dc2626",
                fontweight="bold",
            )
            draw_edges(child)

    def draw_nodes(node):
        if node is None:
            return

        x, y = positions[node]
        is_leaf = node.is_leaf()
        face_color = "#dbeafe" if is_leaf else "#f3f4f6"
        edge_color = "#2563eb" if is_leaf else "#374151"

        ax.scatter(
            [x],
            [y],
            s=950 if is_leaf else 760,
            c=face_color,
            edgecolors=edge_color,
            linewidths=1.5,
            zorder=3,
        )
        ax.text(
            x,
            y,
            huffman_node_label(node),
            ha="center",
            va="center",
            fontsize=9,
            zorder=4,
        )

        draw_nodes(node.left)
        draw_nodes(node.right)

    draw_edges(root)
    draw_nodes(root)

    xs = [pos[0] for pos in positions.values()]
    ys = [pos[1] for pos in positions.values()]
    ax.set_xlim(min(xs) - 1, max(xs) + 1)
    ax.set_ylim(min(ys) - 1, max(ys) + 1)

    plt.tight_layout()
    plt.show()


draw_huffman_tree(huffman_root)

## 7. 哈夫曼编码长度分析

下面计算哈夫曼编码的平均码长、信源熵和压缩效果。

本实验给出三个常用指标：

1. **压缩后比例**：`哈夫曼编码长度 / 固定长度编码长度`，数值越小表示压缩后越短。
2. **压缩倍数**：`固定长度编码长度 / 哈夫曼编码长度`，数值越大表示压缩效果越明显。
3. **节省比例**：`1 - 哈夫曼编码长度 / 固定长度编码长度`，表示节省了多少编码长度。

In [ ]:
def huffman_average_code_length(frequencies, codes):
    """
    计算哈夫曼编码平均码长。
    """
    total = sum(frequencies.values())
    length_sum = 0
    for symbol, frequency in frequencies.items():
        length_sum += frequency * len(codes[symbol])
    return length_sum / total


def entropy(frequencies):
    """
    计算符号熵。
    """
    total = sum(frequencies.values())
    value = 0.0
    for frequency in frequencies.values():
        p = frequency / total
        value -= p * math.log2(p)
    return value


def compression_statistics(original_bits, compressed_bits):
    """
    计算压缩相关指标。

    参数：
        original_bits: 压缩前编码长度
        compressed_bits: 压缩后编码长度

    返回：
        compressed_ratio: 压缩后比例，compressed_bits / original_bits
        compression_factor: 压缩倍数，original_bits / compressed_bits
        saving_ratio: 节省比例，1 - compressed_bits / original_bits
    """
    compressed_ratio = compressed_bits / original_bits
    compression_factor = original_bits / compressed_bits
    saving_ratio = 1 - compressed_ratio
    return compressed_ratio, compression_factor, saving_ratio


fixed_length_bits = math.ceil(math.log2(len(frequencies))) * len(sample_text)
huffman_length = len(huffman_bits)
huffman_ratio, huffman_factor, huffman_saving = compression_statistics(
    fixed_length_bits,
    huffman_length,
)

print("符号种类数：", len(frequencies))
print("固定长度编码总长度：", fixed_length_bits, "bit")
print("哈夫曼编码总长度：", huffman_length, "bit")
print("哈夫曼平均码长：", huffman_average_code_length(frequencies, huffman_codes))
print("信源熵：", entropy(frequencies))
print("哈夫曼压缩后比例：", f"{huffman_ratio:.4f}")
print("哈夫曼压缩倍数：", f"{huffman_factor:.4f}")
print("哈夫曼节省比例：", f"{huffman_saving * 100:.2f}%")

## 8. 实现算术编码

In [ ]:
def build_arithmetic_intervals(frequencies):
    """
    TODO：根据频率表构建算术编码的累积概率区间。
    """
    # TODO 1：计算总频数 total。
    # TODO 2：设置 cumulative = Fraction(0, 1)。
    # TODO 3：按 sorted(frequencies) 遍历符号。
    # TODO 4：计算当前符号概率 probability = Fraction(count, total)。
    # TODO 5：为符号分配 [cumulative, cumulative + probability)。
    # TODO 6：更新 cumulative。
    # TODO 7：返回 intervals。
    raise NotImplementedError("请补全 build_arithmetic_intervals 函数")


def arithmetic_encode(text, intervals):
    """
    TODO：算术编码，将整个文本编码为 [low, high) 区间中的一个数。
    """
    # TODO 1：初始化 low = 0, high = 1。
    # TODO 2：遍历 text 中的每个 symbol。
    # TODO 3：取出 symbol 的全局概率区间 [symbol_low, symbol_high)。
    # TODO 4：根据当前区间宽度更新 new_low 和 new_high。
    # TODO 5：全部字符处理完后，选择区间中点 tag。
    # TODO 6：返回 tag, low, high。
    raise NotImplementedError("请补全 arithmetic_encode 函数")


def arithmetic_decode(tag, message_length, intervals):
    """
    TODO：算术解码。
    """
    # TODO 1：设置 value = tag。
    # TODO 2：重复 message_length 次。
    # TODO 3：查找 value 落入哪个符号区间。
    # TODO 4：输出该符号。
    # TODO 5：将 value 归一化到该符号子区间内部。
    # TODO 6：返回解码字符串。
    raise NotImplementedError("请补全 arithmetic_decode 函数")


arithmetic_intervals = build_arithmetic_intervals(frequencies)
arithmetic_tag, arithmetic_low, arithmetic_high = arithmetic_encode(sample_text, arithmetic_intervals)
arithmetic_decoded = arithmetic_decode(arithmetic_tag, len(sample_text), arithmetic_intervals)

print("算术编码符号区间：")
for symbol, (low, high) in arithmetic_intervals.items():
    print(f"{repr(symbol):>4} -> [{float(low):.6f}, {float(high):.6f})")

print("\n最终编码区间：")
print("low  =", arithmetic_low)
print("high =", arithmetic_high)
print("tag  =", arithmetic_tag)
print("\n算术解码是否正确：", arithmetic_decoded == sample_text)

## 9. 将算术编码区间转换为二进制码字

In [ ]:
def find_binary_fraction_in_interval(low, high, max_bits=300):
    """
    TODO：在 [low, high) 区间内寻找一个尽可能短的二进制小数。

    返回：
        bits: 二进制小数字符串，不含小数点
        value: bits 对应的 Fraction 码值
    """
    # TODO 1：从 bit_length = 1 开始逐渐增加。
    # TODO 2：令 denominator = 2 ** bit_length。
    # TODO 3：找到第一个使 numerator / denominator >= low 的 numerator。
    # TODO 4：构造 value = Fraction(numerator, denominator)。
    # TODO 5：如果 low <= value < high，则返回二进制 bits 和 value。
    # TODO 6：如果超过 max_bits 仍未找到，抛出 ValueError。
    raise NotImplementedError("请补全 find_binary_fraction_in_interval 函数")


arithmetic_bits, arithmetic_binary_tag = find_binary_fraction_in_interval(
    arithmetic_low,
    arithmetic_high,
    max_bits=300,
)
decoded_from_binary_tag = arithmetic_decode(
    arithmetic_binary_tag,
    len(sample_text),
    arithmetic_intervals,
)

print("算术编码二进制码字：")
print(arithmetic_bits)
print("算术编码二进制长度：", len(arithmetic_bits), "bit")
print("使用二进制码值解码是否正确：", decoded_from_binary_tag == sample_text)

## 10. 哈夫曼编码与算术编码对比

In [ ]:
print("编码方法       | 编码长度(bit) | 是否正确解码")
print("-" * 42)
print(f"固定长度编码   | {fixed_length_bits:>12} | -")
print(f"哈夫曼编码     | {len(huffman_bits):>12} | {huffman_decoded == sample_text}")
print(f"算术编码       | {len(arithmetic_bits):>12} | {decoded_from_binary_tag == sample_text}")

print("\n说明：")
print("1. 哈夫曼编码为每个符号分配一个前缀码。")
print("2. 算术编码将整条消息映射为一个区间内的数。")
print("3. 算术编码通常更接近熵极限，但实现复杂度更高。")
print("4. 本实验使用 Fraction 精确演示区间计算，实际工程实现会使用整数区间和重归一化技术。")

## 11. 实验小结

本实验手写实现了哈夫曼编码和算术编码。

需要掌握的重点：

1. 哈夫曼编码根据频率构造二叉树，高频符号获得短码，低频符号获得长码。
2. 哈夫曼编码是前缀码，可以逐位扫描并唯一解码。
3. 算术编码将整条消息表示为一个逐步缩小的区间。
4. 算术编码最终可以选择区间中的一个二进制小数作为码字。
5. 哈夫曼编码实现相对直观，算术编码更接近概率模型本身。
6. 本实验没有调用第三方编码库，编码表、区间更新、编码和解码均手写实现。

思考题：

1. 为什么高频符号应该分配更短的编码？
2. 哈夫曼编码为什么要求不能有某个码字是另一个码字的前缀？
3. 算术编码为什么需要知道消息长度或终止符？
4. 当符号概率非常不均匀时，哈夫曼编码和算术编码的长度差异会如何变化？